# Small chatbot with DistilGPT2

Fine-tuning DistilGPT2 on a CSV of question and answer pairs so it answers like a simple chatbot. I added a `<bot>:` separator token and masked the question part so the loss is only on the reply. No metric here, just the training loss per epoch and a couple of sample replies.

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import os

# 1. Load the tokenizer and model weights
model_name = "distilgpt2" # You can also use "gpt2-medium", "gpt2-large", or "gpt2-xl"
model = AutoModelForCausalLM.from_pretrained(model_name)

# 2. Check if a GPU is available in Colab and move the model to it
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"Model successfully loaded onto: {device}")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model successfully loaded onto: cuda


In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_name, pad_token = "<pad>", bos_token = "<start>", eos_token = "<end>")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.add_tokens(["<bot>:"])

1

In [11]:
from torch.utils.data import Dataset
import pandas as pd

class CustomDataset(Dataset):
  def __init__(self, path):
    df = pd.read_csv(path)
    self.df = df
    self.answers = df.loc[:, 'answer']
    self.questions = df.loc[:, 'question']

    self.input_ids = []
    self.attention_masks = []

    for i in range(len(self.df)):
      text = "<start> " + self.questions[i] + " <bot>: " + self.answers[i] + " <end>"
      encoded = tokenizer(text,
                          truncation = True,
                          max_length=64,
                          padding = "max_length",
                          return_tensors = "pt")

      self.input_ids.append(encoded["input_ids"].flatten(0))
      self.attention_masks.append(encoded["attention_mask"].flatten(0))

  def __len__(self):
    return len(self.df)

  def __getitem__(self, idx):
    return (self.input_ids[idx], self.attention_masks[idx])

In [13]:
data = CustomDataset("/kaggle/input/datasets/nikolozkiladze/conversation/Conversation.csv")
model.resize_token_embeddings(len(tokenizer))

Embedding(50261, 768)

In [14]:
from torch.utils.data import DataLoader

packed = DataLoader(data, batch_size = 16, shuffle = True)

In [15]:
from torch.optim import AdamW
from transformers import get_scheduler
learning_rate = 2e-5
epochs = 8


optimizer = AdamW(model.parameters(), lr=learning_rate, eps = 1e-8)

scheduler = get_scheduler("linear", optimizer, num_warmup_steps = len(packed)*epochs*0.1, num_training_steps = len(packed)*epochs)

In [27]:
def infer(text1):
  text1 = "You like videos?"
  edited = "<start> " + text1 + " <bot>: "
  bim = "<start> " + text1
  inputs = tokenizer(edited, return_tensors="pt").to('cuda')
  with torch.no_grad():
      output = model.generate(
          **inputs,
          max_new_tokens=50,
          top_k = 20,
          do_sample=True,
          eos_token_id=tokenizer.eos_token_id,  # <-- Tell it EXACTLY when to stop!
          pad_token_id=tokenizer.eos_token_id,
          temperature = 1
      )

  return tokenizer.decode(output)[0][len(bim)+1:]

In [18]:
import tqdm
def train(chatData, model, optim):
    epochs = 5

    # Wrap your dataset in a DataLoader to create batches (e.g., batch_size=4 or 8)
    # Note: Ensure your sequences in chatData are padded to the same length for batching!
    train_loader = DataLoader(chatData, batch_size=4, shuffle=True)

    for i in tqdm.tqdm(range(epochs)):
        for X, a in train_loader:  # Iterates through batches now, not single elements
            X = X.to(device)
            a = a.to(device)

            optim.zero_grad()

            # The shapes are now correctly [batch_size, sequence_length]
            loss = model(input_ids=X, attention_mask=a, labels=X).loss

            loss.backward()
            optim.step()

        torch.save(model.state_dict(), "model_state.pt")
        print(infer("hello how are you"))

In [19]:
def train2(data, model, epochs):
  for epoch in tqdm.tqdm(range(epochs)):
    total_train_loss = 0
    model.train()
    for step, batch in enumerate(packed):
      b_input_ids = batch[0].to('cuda')
      b_masks = batch[1].to('cuda')

      b_labels = b_input_ids.clone()

      # 1. Create a boolean mask where the special token occurs
      is_special = (b_input_ids == 50260)

      # 2. Use cumulative sum to find everything before the first special token
      # Before the token, cumsum is 0. At and after the token, cumsum >= 1.
      mask_before_token = (is_special.cumsum(dim=1) == 0)

      # 3. Mask out everything before the token, AND the token itself
      b_labels[mask_before_token | is_special] = -100

      # 4. Mask out padding tokens
      b_labels[b_labels == tokenizer.pad_token_id] = -100

      optimizer.zero_grad()

      #print("hi")

      outputs = model(input_ids = b_input_ids, labels = b_labels, attention_mask = b_masks, token_type_ids = None)

      #print("bye")

      loss = outputs.loss # Access loss directly from the outputs object when labels are provided.

      loss.backward()
      optimizer.step()
      scheduler.step()
      total_train_loss += loss.item()
    print(total_train_loss)

In [20]:
train2(data, model, 10)

 10%|█         | 1/10 [00:45<06:47, 45.29s/it]

727.0623168945312


 20%|██        | 2/10 [01:28<05:52, 44.04s/it]

627.4847977161407


 30%|███       | 3/10 [02:12<05:08, 44.14s/it]

592.991537809372


... (10 outputs left out)

 90%|█████████ | 9/10 [06:35<00:43, 43.80s/it]

519.4777457714081


100%|██████████| 10/10 [07:19<00:00, 43.92s/it]

518.38492000103


In [30]:
model.eval()
with torch.no_grad():
  print(infer("What are you doing?"))

<bot>: Â

Â
Â
Â
Â
Â
Â
Â
Â
Â
Â
Â
Â
Â
Â
Â
Â
Â
Â
Â
Â
Â
Â
Â
Â


In [60]:
def infer1(text1, max_new_tokens=50):
    # 1. Ensure model is in eval mode
    model.eval()
    edited = "<start> " + text1 + " <bot>: "
    bim = "<start> " + text1
    # 2. Configure tokenizer to left-pad for generation
    tokenizer.padding_side = "left"
    print(edited)
    # 3. Tokenize input with attention mask
    inputs = tokenizer(edited, return_tensors="pt", padding=True)
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)
    
    # 4. Generate with safe sampling parameters
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=True,          # Breaks the greedy repetition loop
            top_k=20,                # Filters out low-probability chaos tokens
            top_p=0.92,              # Nucleus sampling for natural flow
            temperature=0.7,         # Lowers randomness to keep it coherent
            repetition_penalty=1.2,  # Actively penalizes repeating characters like Â
            pad_token_id=tokenizer.eos_token_id, # Crucial for GPT-2
            eos_token_id=tokenizer.eos_token_id
        )
        
    # 5. Decode only the newly generated tokens
    # This strips out your prompt and any padding attached to it
    new_tokens = output_ids[0][input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [62]:
infer1("What is your favorite car?")

<start> What is your favorite car? <bot>: 


'Â i like to drive around the world. what do you have in common with yours? Â well, there are two things that stand out for me: one being a great mechanic and another having fun driving my own cars! \xa0 all three'

In [42]:
tokenizer.decode(data[0][0])

"<start> hi, how are you doing? <bot>: i'm fine. how about yourself? <end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end><end>"